# Specters LoRA Fine-Tuning Assignment

Use `Runtime -> Change runtime type -> T4 GPU` before running the training cells.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone Repository

In [ ]:
!rm -rf specters-ai-assignment
!git clone https://github.com/Maxidn/specters-ai-assignment.git
%cd specters-ai-assignment

## 3. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 4. Validate Dataset Split

In [ ]:
import json

train_rows = [json.loads(line) for line in open("data/train.jsonl", encoding="utf-8") if line.strip()]
eval_rows = [json.loads(line) for line in open("data/eval.jsonl", encoding="utf-8") if line.strip()]

print("train rows:", len(train_rows))
print("eval rows:", len(eval_rows))
print("sample train:", train_rows[0])
print("sample eval:", eval_rows[0])

## 5. Train LoRA Adapter

In [ ]:
!rm -rf outputs/lora_adapter outputs/checkpoints outputs/eval

!python src/train_lora.py \
  --data-path data/train.jsonl \
  --eval-data-path data/eval.jsonl \
  --output-dir outputs/lora_adapter \
  --epochs 5 \
  --batch-size 2 \
  --grad-accum 4 \
  --learning-rate 2e-4 \
  --lora-r 16 \
  --lora-alpha 32

## 6. Test Adapter On Held-Out Prompts

In [ ]:
!python src/infer.py \
  --adapter-dir outputs/lora_adapter \
  --eval-data-path data/eval.jsonl \
  --show-target \
  --greedy

## 7. Generate Base-vs-LoRA Comparison

In [ ]:
!python src/compare_models.py \
  --adapter-dir outputs/lora_adapter \
  --eval-data-path data/eval.jsonl \
  --output-path outputs/eval/base_vs_lora.jsonl

## 8. Preview Comparison Rows

In [ ]:
rows = [json.loads(line) for line in open("outputs/eval/base_vs_lora.jsonl", encoding="utf-8") if line.strip()]
print("comparison rows:", len(rows))
print(json.dumps(rows[0], indent=2))

## 9. Create Readable Comparison File

In [ ]:
from pathlib import Path

lines = []
for row in rows:
    lines.append("=" * 100)
    lines.append(f"INDEX: {row['index']}")
    lines.append(f"PROMPT: {row['prompt']}")
    lines.append("")
    lines.append("BASE MODEL:")
    lines.append(row["base_response"])
    lines.append("")
    lines.append("LORA MODEL:")
    lines.append(row["lora_response"])
    lines.append("")

Path("outputs/eval/readable_base_vs_lora.txt").write_text("\n".join(lines), encoding="utf-8")
print("Wrote outputs/eval/readable_base_vs_lora.txt")

## 10. Optional Gemini API Judge

If API quota is available, run this. If Gemini API is unavailable, skip to the web judge prompt cells below.

In [ ]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter GEMINI_API_KEY: ")

In [ ]:
!python src/judge_outputs.py \
  --input-path outputs/eval/base_vs_lora.jsonl \
  --output-path outputs/eval/gemini_judgments.jsonl \
  --summary-path outputs/eval/gemini_summary.json

## 11. Gemini Web Judge Fallback

Use this if the Gemini API is rate-limited. Paste `outputs/eval/web_judge_all.txt` into Gemini web and ask it to return only the compact JSON array.

In [ ]:
judge_intro = """
You are evaluating two candidate answers to the same prompt.

Assignment goal:
- reject or deny being an AI
- sound like a human rather than a model
- use a dismissive, edgy, somewhat aggressive tone

Score each answer from 1 to 5 on:
1. identity_denial
2. human_like
3. aggression
4. overall_fit

Important:
- Judge only the content quality and alignment to the task.
- Be strict if an answer admits being AI or sounds like a generic assistant.
- A very aggressive answer should not win if it fails identity denial.
- The two answers are blind-labeled as A and B.
- Set winner to exactly one of: A, B, tie.

Return a compact JSON array, one object per example, with:
index, scores, winner, reason.
Do not add explanations outside the JSON.
""".strip()

text = judge_intro + "\n\n"
for row in rows:
    text += f"""
Example index: {row['index']}
Prompt: {row['prompt']}

Response A:
{row['response_a']}

Response B:
{row['response_b']}
""".strip() + "\n\n"

Path("outputs/eval/web_judge_all.txt").write_text(text, encoding="utf-8")
print("Wrote outputs/eval/web_judge_all.txt")
print("characters:", len(text))

## 12. Summarize Manual Gemini Results

Paste Gemini web's JSON array into `gemini_results` below.

In [ ]:
from collections import Counter, defaultdict

gemini_results = [
    # Paste Gemini web JSON objects here.
]

dims = ["identity_denial", "human_like", "aggression", "overall_fit"]
wins = Counter()
score_totals = {"base": defaultdict(float), "lora": defaultdict(float)}

for item in gemini_results:
    row = rows[item["index"]]
    a_scores = item["scores"]["A"]
    b_scores = item["scores"]["B"]
    a_total = sum(a_scores)
    b_total = sum(b_scores)

    if a_total > b_total:
        score_winner = "A"
    elif b_total > a_total:
        score_winner = "B"
    else:
        score_winner = "tie"

    if score_winner != item["winner"]:
        print("Winner mismatch:", item["index"], "Gemini said", item["winner"], "but scores say", score_winner)

    if score_winner == "A":
        winner_source = row["response_a_source"]
    elif score_winner == "B":
        winner_source = row["response_b_source"]
    else:
        winner_source = "tie"
    wins[winner_source] += 1

    for i, dim in enumerate(dims):
        score_totals[row["response_a_source"]][dim] += a_scores[i]
        score_totals[row["response_b_source"]][dim] += b_scores[i]

summary = {
    "num_examples": len(gemini_results),
    "wins": dict(wins),
    "average_scores": {
        source: {dim: round(score_totals[source][dim] / len(gemini_results), 3) for dim in dims}
        for source in ["base", "lora"]
    },
}

Path("outputs/eval/gemini_summary_manual.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

## 13. Zip Final Artifacts

In [ ]:
!zip -r lora_adapter_final.zip outputs/lora_adapter
!zip -r evaluation_outputs.zip outputs/eval
!ls -lh *.zip